In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

Se importan las librerías necesarias para trabajar los datos.

In [2]:
ruta_archivo = "https://raw.githubusercontent.com/Daniel-UTEC/bermudez_daniel_2503162022/refs/heads/main/data/clave_A_asociacion.csv"
df = pd.read_csv(ruta_archivo)
df.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,A-T0001,A-C0058,2026-01-07,Limpieza,Detergente,1,App
1,A-T0001,A-C0058,2026-01-07,Panaderia,Pastelito,1,App
2,A-T0001,A-C0058,2026-01-07,Limpieza,Suavizante,1,App
3,A-T0002,A-C0044,2026-02-14,Panaderia,Pastelito,3,Web
4,A-T0003,A-C0002,2026-03-19,Lacteos,Queso,2,Web


Se carga la data raw almacenada en el repositorio de GitHub.
Luego, se muestran las primeras 5 filas de la data, lo que nos da a entender que la información puede tratarse de los registros de salida de un supermercado o una tienda.
Contiene variables mayoritariamente de texto y algunas variables numéricas.

In [3]:
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580 entries, 0 to 579
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  580 non-null    object
 1   cliente_id      580 non-null    object
 2   fecha           580 non-null    object
 3   categoria       580 non-null    object
 4   item            580 non-null    object
 5   cantidad        580 non-null    int64 
 6   canal           579 non-null    object
dtypes: int64(1), object(6)
memory usage: 31.8+ KB


Se puede observar que todos las columnas de la información son de tipo texto, a excepción de la columna de cantidad que es de tipo numérica.

In [5]:
df.isnull().sum()

,0
transaccion_id,0
cliente_id,0
fecha,0
categoria,0
item,0
cantidad,0
canal,1


In [6]:
df.duplicated().sum()

np.int64(1)

Al validar si en las columnas existen campos nulos, se puede observar que únicamente hay un valor nulo en la columna canal. Este registro no será eliminado puesto que puede afectar el resultado final, dado que corresponde a una salida de producto que no se puede omitir.


Respecto a los valores duplicados, se tiene únicamente un valor duplicado.

In [8]:
df[df.duplicated()]

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
579,A-T0005,A-C0073,2026-03-18,Lacteos,Queso,1,Tienda


Al revisar el valor duplicado, se puede observar que debido a la columna transacción no es posible tener registros duplicados, puesto que si hiciera referencia a otra compra, tendría un número de transacción diferente. Por ello, se procederá a eliminar el registro.

In [11]:
df.drop_duplicates(inplace=True)

In [13]:
variables_numericas = ["cantidad"]
for columna in variables_numericas:
    media = df[columna].mean()
    desviacion = df[columna].std()
    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion
    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    print(columna, media, desviacion, limite_inferior, limite_superior, len(outliers))


cantidad 1.4162348877374784 0.7448640665277593 -0.07349324531804013 2.905963020792997 58


Según el resultado del código, se puede observar que existen 58 registros atípicos que no coinciden con el rango establecido de limite superior e inferior.

In [15]:
transacciones = df.groupby('transaccion_id')['item'].apply(list).values.tolist()
display(transacciones[:5])

[['Detergente', 'Pastelito', 'Suavizante'],
 ['Pastelito'],
 ['Queso', 'Yogurt'],
 ['Agua', 'Arroz', 'Galletas', 'Pasta', 'Queso', 'Salsa'],
 ['Cafe', 'Leche', 'Pan', 'Queso']]

In [16]:
columnas_asociacion = [
    "transaccion_id", "item"
]

df_asociacion = df[columnas_asociacion].copy()
